In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import edunn as nn
from edunn import utils

# Logistic Regression Model: `forward`

A Logistic Regression model is built by applying the `Softmax` function to a Linear Regression model. This way, this function turns the output vector of the Linear Regression into a vector that represents a probability distribution.

The Logistic Regression function is $f(x)=softmax(wx+b)$. However, just as we did with the `LinearRegression` model, we can view this model as the application of
* A `Linear` layer $f(x)=wx$,
* A `Bias` layer $f(x)=x+b$
* A `Softmax` layer $f(x)=softmax(x)$

That is, we have the following sequence of transformations `x → Linear → Bias → Softmax → y`. 


Implement the `forward` method of the `LogisticRegression` model in the file `edunn/models/logistic_regression.py`. To do so, we have already defined and initialized internal models of class `Linear`, `Bias` and `Softmax`; you only have to call their respective `forward`s in the right order.

In [ ]:
x = np.array([[1,0],
             [0,1],
             [1,1]])

w = np.array([[100,0,0],
              [0,100,0],
              ])
b = np.array([0,0,0])
linear_initializer = nn.initializers.Constant(w)
bias_initializer = nn.initializers.Constant(b)
layer=nn.LogisticRegression(2,3,linear_initializer=linear_initializer,bias_initializer=bias_initializer)
y = np.array([[1, 0,0],
              [0, 1,0],
              [0.5,0.5,0]
             ])

utils.check_same(y,layer.forward(x))

y = np.array([[0, 0.5,0.5],
              [0.5, 0,0.5],
              [0,0,1]
             ])
linear_initializer = nn.initializers.Constant(-w)
bias_initializer = nn.initializers.Constant(-b)
layer=nn.LogisticRegression(2,3,linear_initializer=linear_initializer,bias_initializer=bias_initializer)
utils.check_same(y,layer.forward(x))



# Logistic Regression Model: `backward`

The `backward` method of a `LogisticRegression` model is the *reverse* composition of the `backward` functions of the `Linear`, `Bias`, and `Softmax` layers. Remember that these are applied in the opposite order from the forward method.

In this case, we also help you out by combining each layer's gradient dictionary into a single big dictionary of `LogisticRegression` gradients, using the `**` operator that unpacks a dictionary, together with `{**dict1, **dict2}` which merges them back together.

Implement the `backward` method of the `LogisticRegression` model:


In [ ]:
samples = 100
batch_size=2
din=3 # input dimension
dout=5 # output dimension
input_shape=(batch_size,din)

# Check the derivatives of a Logistic Regression model
# with random values for `w`, `b`, and `x`, the input
layer=nn.LogisticRegression(din,dout)

utils.check_gradient.common_layer(layer,input_shape,samples=samples,tolerance=1e-5)    


# Applied Logistic Regression

Now that we have all the pieces, we can define and train our first logistic regression model to classify the flowers of the [Iris](https://www.kaggle.com/uciml/iris) dataset.

In this case, we will train the model with the mean squared error function; however, while this form of the error will work for this problem, it makes the optimization problem non-_convex_, and therefore there is no single global minimum. Later on, we will implement the _Cross-Entropy_ error function, designed specifically to deal with outputs that represent probability distributions.

In [ ]:
import edunn as nn
import numpy as np
from edunn import metrics,datasets

# Load the data, where `y` uses "onehot" encoding
# `y` has as many columns as there are classes
# if example i belongs to class 2, for instance, then
# y[i,2]=1 and the rest of the values of y[i,:] are 0
# (note: class labels start at 0)
x,y,classes=datasets.load_classification("iris",onehot=True)
# data normalization
x = (x-x.mean(axis=0))/x.std(axis=0)
n, din = x.shape
n, dout = y.shape

print("Dataset sizes:", x.shape,y.shape)

#Logistic regression model
model = nn.LogisticRegression(din,dout)
# Mean squared error
error = nn.MeanError(nn.SquaredError())
optimizer = nn.GradientDescent(lr=0.1,epochs=1000,batch_size=32)

# Optimization algorithm
history = optimizer.optimize(model,x,y,error)
nn.plot.plot_history(history,error_name=error.name)


print("Model error:")
y_pred=model.forward(x)
metrics.classification_summary_onehot(y,y_pred)
